# 🔬 Notebook 3: Deep Dive — Priority, Retries, Idempotency

For each topic we show the same **bad → better → best** pattern so you can *feel* why each improvement exists. Every cell runs in pure Python — no external services.

## 🛠️ Setup

```bash
cd 06-system-designs/notification-system
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 1. Priority queues

Scenario: the marketing team pushes **1,000,000** promo messages. A second later, a user
requests a 2FA code. How do we make sure the 2FA code doesn't wait in line behind a million
promos?

### ❌ Bad — a single FIFO queue

Everything shares one line. High-priority messages are stuck behind low-priority ones.

In [1]:
from collections import deque

q = deque()
for i in range(1_000_000): q.append(("low",  f"promo-{i}"))
q.append(("high", "2FA-code-for-alice"))

# Pop the first 3 — the 2FA code is a million items away.
print("first 3 popped:", [q.popleft() for _ in range(3)])
print("index of 2FA code in queue:", 1_000_000 - 3)


first 3 popped: [('low', 'promo-0'), ('low', 'promo-1'), ('low', 'promo-2')]
index of 2FA code in queue: 999997


### ⚠️ Better — strict priority (always drain high first)

Multiple queues, and the worker *always* pulls `high` before `normal` before `low`.
Problem: if `high` traffic is continuous, **low/normal starve forever**.

In [2]:
from collections import deque

qs = {"high": deque(), "normal": deque(), "low": deque()}
for i in range(5):  qs["low"].append(f"promo-{i}")
for i in range(3):  qs["normal"].append(f"receipt-{i}")
# Continuous high-priority stream:
for i in range(10): qs["high"].append(f"2fa-{i}")

def strict_pop():
    for level in ("high","normal","low"):
        if qs[level]:
            return level, qs[level].popleft()
    return None

order = []
for _ in range(12):
    order.append(strict_pop())
print(order)
print("⚠️  receipts and promos never got picked while high had items.")


[('high', '2fa-0'), ('high', '2fa-1'), ('high', '2fa-2'), ('high', '2fa-3'), ('high', '2fa-4'), ('high', '2fa-5'), ('high', '2fa-6'), ('high', '2fa-7'), ('high', '2fa-8'), ('high', '2fa-9'), ('normal', 'receipt-0'), ('normal', 'receipt-1')]
⚠️  receipts and promos never got picked while high had items.


### ✅ Best — weighted round-robin (starvation-free)

Pull a fixed **ratio** per round: e.g. 4 high : 2 normal : 1 low.
Low priority still makes progress, but always loses the fight for throughput.

In [3]:
from collections import deque

class WeightedScheduler:
    def __init__(self, ratio):
        self.queues = {k: deque() for k in ratio}
        self.ratio  = ratio
    def push(self, level, msg): self.queues[level].append(msg)
    def drain_round(self):
        out = []
        for level, n in self.ratio.items():
            for _ in range(n):
                if self.queues[level]:
                    out.append((level, self.queues[level].popleft()))
        return out

s = WeightedScheduler({"high":4, "normal":2, "low":1})
for i in range(10): s.push("low",    f"promo-{i}")
for i in range(5):  s.push("normal", f"txn-{i}")
for i in range(3):  s.push("high",   f"2fa-{i}")

rounds = 0
while any(s.queues.values()):
    rounds += 1
    print(f"round {rounds}: {s.drain_round()}")
print("✅ all three levels made progress every round.")


round 1: [('high', '2fa-0'), ('high', '2fa-1'), ('high', '2fa-2'), ('normal', 'txn-0'), ('normal', 'txn-1'), ('low', 'promo-0')]
round 2: [('normal', 'txn-2'), ('normal', 'txn-3'), ('low', 'promo-1')]
round 3: [('normal', 'txn-4'), ('low', 'promo-2')]
round 4: [('low', 'promo-3')]
round 5: [('low', 'promo-4')]
round 6: [('low', 'promo-5')]
round 7: [('low', 'promo-6')]
round 8: [('low', 'promo-7')]
round 9: [('low', 'promo-8')]
round 10: [('low', 'promo-9')]
✅ all three levels made progress every round.


## 2. Retries

Scenario: Twilio returns `503 Service Unavailable`. We want to retry — but *how* matters a lot.

### ❌ Bad — immediate retry loop

Hammers the already-struggling provider, makes the outage worse (**thundering herd**).

In [4]:
import random
random.seed(4)   # a seed where the first few attempts fail, for a clearer demo

def flaky_send():                         # pretend provider: ~70% fail
    return "OK" if random.random() > 0.7 else "FAIL"

attempts = 0
while True:
    attempts += 1
    if flaky_send() == "OK":
        print(f"succeeded on attempt {attempts}"); break
    print(f"  attempt {attempts} failed → retry IMMEDIATELY (no delay)")
    if attempts >= 8:
        print("gave up"); break
print(f"❌ hit the provider {attempts} times back-to-back — thundering herd")


  attempt 1 failed → retry IMMEDIATELY (no delay)
  attempt 2 failed → retry IMMEDIATELY (no delay)
  attempt 3 failed → retry IMMEDIATELY (no delay)
  attempt 4 failed → retry IMMEDIATELY (no delay)
  attempt 5 failed → retry IMMEDIATELY (no delay)
  attempt 6 failed → retry IMMEDIATELY (no delay)
succeeded on attempt 7
❌ hit the provider 7 times back-to-back — thundering herd


### ⚠️ Better — exponential backoff

Wait longer between attempts: `1s, 2s, 4s, 8s, …`. Gentler on the provider, but if
1,000 workers all failed at the same moment they'll *all* retry at t=1s, t=2s, … —
a synchronized stampede.

In [5]:
import random, time
random.seed(3)   # seed that fails a few times first to show the backoff

def backoff_exp(attempt, base=0.05, cap=1.0):     # no jitter
    return min(cap, base * (2 ** attempt))

MAX = 8
start = time.time()
attempts = 0
while attempts < MAX:
    if flaky_send() == "OK":
        print(f"OK after {attempts+1} attempts, elapsed {time.time()-start:.2f}s"); break
    wait = backoff_exp(attempts)
    print(f"  attempt {attempts+1} failed → sleep {wait:.3f}s (doubles every time)")
    time.sleep(wait)
    attempts += 1


  attempt 1 failed → sleep 0.050s (doubles every time)


  attempt 2 failed → sleep 0.100s (doubles every time)


  attempt 3 failed → sleep 0.200s (doubles every time)


  attempt 4 failed → sleep 0.400s (doubles every time)


  attempt 5 failed → sleep 0.800s (doubles every time)


  attempt 6 failed → sleep 1.000s (doubles every time)


  attempt 7 failed → sleep 1.000s (doubles every time)


OK after 8 attempts, elapsed 4.55s


### ✅ Best — exponential backoff **with full jitter** + **DLQ**

Two fixes:

1. **Jitter**: sleep a *random* time in `[0, exp_wait]` so retries spread out.
2. **Dead-letter queue**: after N failed attempts, park the message for human review
   instead of retrying forever.

In [6]:
import random, time
random.seed(1)

def backoff_jitter(attempt, base=0.01, cap=1.0):
    return random.uniform(0, min(cap, base * (2 ** attempt)))

MAX_ATTEMPTS = 5
dlq = []

def send_with_retry(msg):
    for attempt in range(MAX_ATTEMPTS):
        if flaky_send() == "OK":
            return f"{msg}: OK on attempt {attempt+1}"
        time.sleep(backoff_jitter(attempt))
    dlq.append(msg)
    return f"{msg}: → DLQ after {MAX_ATTEMPTS}"

for m in ["m1","m2","m3","m4","m5"]:
    print(send_with_retry(m))
print("DLQ:", dlq)
print("✅ protected provider, bounded retries, ops can inspect DLQ")


m1: OK on attempt 2


m2: OK on attempt 3
m3: OK on attempt 2


m4: OK on attempt 3


m5: OK on attempt 2
DLQ: []
✅ protected provider, bounded retries, ops can inspect DLQ


## 3. Idempotency

Scenario: caller's HTTP client times out after 30s, so it retries. Our service actually
received the first request and queued it. Without idempotency we send **two** receipts.

### ❌ Bad — no dedup, just send

User gets two "Your order shipped!" push notifications. 🤦

In [7]:
sent = []
def naive_send(payload): sent.append(payload)

naive_send({"user":42,"msg":"order shipped"})
naive_send({"user":42,"msg":"order shipped"})   # caller retry
print("sent:", sent)
print(f"❌ user got {len(sent)} copies")


sent: [{'user': 42, 'msg': 'order shipped'}, {'user': 42, 'msg': 'order shipped'}]
❌ user got 2 copies


### ⚠️ Better — hash the payload

Hash the whole payload and skip duplicates. Works for accidental retries, but:

- Any tiny difference (e.g. timestamp field) → different hash → still duplicates.
- Intentional resends (customer clicked "resend receipt") get silently dropped.

In [8]:
import hashlib, json
seen_hashes = set()
def hash_send(payload):
    h = hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()
    if h in seen_hashes: return "duplicate"
    seen_hashes.add(h)
    return "sent"

p = {"user":42, "msg":"order shipped"}
print(hash_send(p))                                    # sent
print(hash_send(p))                                    # duplicate ✔
print(hash_send({**p, "ts": 1700000000}))              # ❌ timestamp makes it "new"


sent
duplicate
sent


### ✅ Best — caller-provided `dedup_key` with TTL

The caller picks a **meaningful** key (`order-123-shipped`). We store it with a TTL long
enough to cover any reasonable retry window (e.g. 7 days). Same key → no-op. Different
key → genuinely different send, even if the payload looks similar.

In [9]:
import time

class DedupStore:
    def __init__(self, ttl_seconds=7*24*3600):
        self.ttl = ttl_seconds
        self._d  = {}                                  # key -> (first_seen_ts, payload)

    def _evict(self):
        now = time.time()
        for k in [k for k,(ts,_) in self._d.items() if now - ts > self.ttl]:
            del self._d[k]

    def enqueue(self, dedup_key, payload):
        self._evict()
        if dedup_key in self._d:
            return "DUPLICATE — not re-sent"
        self._d[dedup_key] = (time.time(), payload)
        return "NEW — queued"

store = DedupStore()
print(store.enqueue("order-123-shipped", {"m":"Your order shipped!"}))
print(store.enqueue("order-123-shipped", {"m":"Your order shipped!"}))   # retry
print(store.enqueue("order-123-resent",  {"m":"Your order shipped!"}))   # support resent
print(store.enqueue("order-124-shipped", {"m":"Different order!"}))
print("✅ retries dedupe, intentional resends get through")


NEW — queued
DUPLICATE — not re-sent
NEW — queued
NEW — queued
✅ retries dedupe, intentional resends get through


## Recap

| Concern | ❌ Bad | ⚠️ Better | ✅ Best |
|---|---|---|---|
| Priority | single FIFO | strict priority | weighted round-robin |
| Retries | tight loop | exponential backoff | exp backoff + jitter + DLQ |
| Idempotency | none | hash payload | caller dedup_key + TTL |

In **Notebook 4** we tackle the remaining production concerns:
**fan-out** (one event → many channels), **per-provider rate limiting**, and **circuit
breakers** to fail fast when a provider is down.